# CharacterTextSplitter

`CharacterTextSplitter`는 가장 기본적인 텍스트 분할 방식입니다.

## 주요 특징

- **단일 구분자 사용**: 기본적으로 `"\n\n"` (이중 줄바꿈)을 기준으로 텍스트를 분할합니다.
- **문자 수 기반**: 청크의 크기를 문자 수(`len()`)로 측정합니다.
- **간단한 구조**: 복잡한 재귀 로직 없이 하나의 구분자만 사용하여 분할합니다.

## 동작 방식

1. **텍스트 분할 기준**: 지정된 단일 구분자 (기본값: `"\n\n"`)
2. **청크 크기 측정**: 문자 수 (`length_function=len`)
3. **청크 중복**: `chunk_overlap` 파라미터로 인접 청크 간 중복 영역 설정

## 사용 시나리오

- 단락 구분이 명확한 텍스트 문서
- 간단한 분할이 필요한 경우
- 특정 구분자를 기준으로 정확히 나누고 싶을 때


# 6-2. Chunking(청킹) — CharacterTextSplitter

## 학습 목표
- 청킹(Chunking)이 RAG에서 왜 필요한지 이해해요
- `CharacterTextSplitter`의 단일 구분자 분할 방식을 익혀요
- `chunk_size`, `chunk_overlap`, `separator` 파라미터의 동작 원리를 파악해요

## 사전 지식
- 6-1 Document Loaders 섹션 완료
- Python 문자열 처리 기본 이해

---

## 문서를 로드했으니, 이제 청킹을 배워요

6-1에서 다양한 형식의 문서를 `Document` 객체로 변환하는 방법을 배웠어요. 그런데 로딩한 문서를 그대로 LLM에 넣을 수는 없어요. **토큰 제한**이 있기 때문이에요.

청킹(Chunking)은 긴 문서를 LLM이 처리할 수 있는 적절한 크기의 조각으로 나누는 작업이에요. RAG 파이프라인에서 두 번째 단계에 해당해요.

```mermaid
flowchart LR
    A[📄 Load<br/>문서 로드]:::process --> B[✂️ Chunk<br/>청크 분할]:::current
    B --> C[🔢 Embed<br/>임베딩 변환]:::process
    C --> D[🗄️ Store<br/>벡터 저장]:::storage
    D --> E[🔍 Retrieve<br/>검색]:::process
    E --> F[💬 Generate<br/>답변 생성]:::output

    classDef current fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

> **CharacterTextSplitter**는 가장 기본적인 분할 방식이에요. 지정한 **단일 구분자**(`\n\n` 등)를 기준으로 텍스트를 나눠요.


> 🔑 **핵심 개념**: 청킹이 RAG 품질의 50%를 결정합니다. 청크가 너무 크면 관련 없는 내용이 섞이고, 너무 작으면 문맥이 끊깁니다. `chunk_size`와 `chunk_overlap`의 균형을 맞추는 것이 핵심입니다.

In [1]:
# 샘플 데이터 파일을 읽어옵니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()


로드된 텍스트의 일부를 확인합니다.


In [2]:
# 파일 내용 중 처음 500자를 출력합니다.
print(file[:500])
print(f"\n[전체 텍스트 길이: {len(file)} 문자]")


Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어

[전체 텍스트 길이: 5733 문자]


## 1. 기본 사용법

`CharacterTextSplitter`의 기본 파라미터를 살펴봅니다.

### 주요 파라미터

- `separator`: 텍스트를 분할할 구분자 (기본값: `"\n\n"`)
- `chunk_size`: 각 청크의 최대 문자 수
- `chunk_overlap`: 인접 청크 간 중복될 문자 수
- `length_function`: 텍스트 길이 계산 함수 (기본값: `len`)


> 🎯 **강의 포인트**: `CharacterTextSplitter`는 **단일 구분자**만 사용하는 가장 단순한 분할기입니다. 지정한 구분자로 나누고, 청크가 `chunk_size`를 초과해도 자동 재분할하지 않습니다. 이 한계 때문에 대부분의 경우 `RecursiveCharacterTextSplitter`를 사용합니다.

In [3]:
# ---------------------------------------------------
# CharacterTextSplitter 생성 및 기본 파라미터 설정
# ---------------------------------------------------

from langchain_text_splitters import CharacterTextSplitter

# 1단계: CharacterTextSplitter 생성
# - separator="\n\n": 이중 줄바꿈(단락 구분)을 기준으로 분할
# - chunk_size=250: 청크 하나의 최대 문자 수
# - chunk_overlap=50: 인접 청크 간 중복 문자 수 (문맥 연결을 위해)
# - length_function=len: 파이썬 기본 len() 함수로 문자 수 측정
text_splitter = CharacterTextSplitter(
    separator="\n\n",  # 이중 줄바꿈을 구분자로 사용
    chunk_size=250,  # 청크 최대 크기: 250자
    chunk_overlap=50,  # 청크 간 중복: 50자
    length_function=len,  # 길이 계산 함수
)

## 2. 텍스트 분할 실행

`create_documents()` 메서드를 사용하여 텍스트를 Document 객체 리스트로 분할합니다.


In [ ]:
# ============================================================
# TODO: text_splitter.create_documents()로 텍스트를 분할해보세요
# 힌트: create_documents([file]) → Document 리스트 반환
# 예상 결과: 30개의 Document가 생성되고 첫 번째 청크의 내용이 출력됩니다
# ============================================================

# 1단계: 텍스트를 Document 객체 리스트로 분할
# - create_documents(): 텍스트 리스트를 입력받아 Document 리스트 반환
documents = text_splitter.create_documents([file])

print(f"생성된 문서 개수: {len(documents)}")
print(f"첫 번째 청크 길이: {len(documents[0].page_content)} 문자")
print("\n" + "=" * 60)
print("첫 번째 청크 내용:")
print("=" * 60)
print(documents[0].page_content)

생성된 문서 개수: 30
첫 번째 청크 길이: 197 문자

첫 번째 청크 내용:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding


두 번째와 세 번째 청크를 확인하여 청크 간 중복(`chunk_overlap`)이 어떻게 작동하는지 살펴봅니다.


> 💡 **실무 팁**: `chunk_overlap`은 `chunk_size`의 10~20% 정도가 적절합니다. 예를 들어 `chunk_size=1000`이면 `chunk_overlap=100~200`으로 설정하세요. 너무 크면 중복 데이터로 벡터 검색 품질이 떨어집니다.

In [ ]:
# 두 번째 청크 확인
print("=" * 60)
print("두 번째 청크:")
print("=" * 60)
print(documents[1].page_content)

print("\n" + "=" * 60)
print("세 번째 청크:")
print("=" * 60)
print(documents[2].page_content)


두 번째 청크:
Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

세 번째 청크:
Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어 처리, 구문 분석

Tokenizer


## 3. 메타데이터 추가

텍스트 분할 시 각 청크에 메타데이터를 추가할 수 있습니다. 이는 문서 추적이나 필터링에 유용합니다.


In [ ]:
# ============================================================
# TODO: create_documents()에 metadatas를 함께 전달해서 청크에 메타데이터를 추가해보세요
# 힌트: create_documents([file, file], metadatas=[{...}, {...}])
# 예상 결과: 각 청크의 metadata에 source, section, version 키가 포함됩니다
# ============================================================

# 1단계: 두 개의 텍스트에 대한 메타데이터 정의
# - 텍스트 수와 메타데이터 수가 일치해야 함
metadatas = [
    {"source": "tech_glossary", "section": "AI", "version": 1},
    {"source": "tech_glossary", "section": "Database", "version": 1},
]

# 2단계: 텍스트와 메타데이터를 함께 전달하여 Document 생성
documents_with_metadata = text_splitter.create_documents(
    [file, file],  # 같은 텍스트를 두 번 사용 (예시용)
    metadatas=metadatas,
)

# 3단계: 메타데이터 확인
print("첫 번째 문서 메타데이터:")
print(documents_with_metadata[0].metadata)
print("\n첫 번째 문서 내용 (일부):")
print(documents_with_metadata[0].page_content[:200] + "...")

첫 번째 문서 메타데이터:
{'source': 'tech_glossary', 'section': 'AI', 'version': 1}

첫 번째 문서 내용 (일부):
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding...


In [ ]:
# 전체 문서 개수 확인
print(f"메타데이터가 포함된 전체 문서 개수: {len(documents_with_metadata)}")

# 중간 문서의 메타데이터 확인 (두 번째 텍스트 소속)
mid_index = len(documents_with_metadata) // 2
print(f"\n{mid_index}번째 문서의 메타데이터:")
print(documents_with_metadata[mid_index].metadata)


메타데이터가 포함된 전체 문서 개수: 60

30번째 문서의 메타데이터:
{'source': 'tech_glossary', 'section': 'Database', 'version': 1}


## 4. split_text() 메서드 사용

`split_text()` 메서드는 Document 객체가 아닌 **순수 문자열 리스트**를 반환합니다.


> ⚠️ **자주 하는 실수**: `CharacterTextSplitter`는 단락 구분이 명확한 문서에만 적합합니다. 줄바꿈이 불규칙하거나 구분자가 없는 텍스트에서는 의도하지 않은 결과가 나옵니다. 그런 경우 `RecursiveCharacterTextSplitter`로 전환하세요.

In [ ]:
# 텍스트를 문자열 리스트로 분할
text_chunks = text_splitter.split_text(file)

print(f"분할된 텍스트 청크 개수: {len(text_chunks)}")
print("\n첫 번째 텍스트 청크:")
print("=" * 60)
print(text_chunks[0])


분할된 텍스트 청크 개수: 30

첫 번째 텍스트 청크:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding


## 5. 다양한 구분자 테스트

구분자를 변경하여 다른 방식으로 텍스트를 분할할 수 있습니다.


## 핵심 정리 및 마무리

### CharacterTextSplitter vs RecursiveCharacterTextSplitter

| 특징 | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|------|:---------------------:|:------------------------------:|
| 구분자 수 | 1개 | 여러 개 (우선순위 순서) |
| 분할 방식 | 단순 분할 | 재귀적 분할 |
| 크기 초과 시 | 그대로 유지 | 자동 재분할 |
| 권장 상황 | 구분자가 명확한 문서 | 대부분의 경우 |

### chunk_overlap의 역할
> `chunk_overlap`은 인접한 청크 사이에 겹치는 텍스트를 만들어요. 청크 경계에서 문맥이 끊기는 것을 방지하기 위해 사용해요. 일반적으로 `chunk_size`의 10~20% 정도로 설정해요.

---

## 다음 예고

CharacterTextSplitter의 한계(단일 구분자, 크기 초과 처리 미흡)를 극복한 방식을 배워볼게요.

- **02-RecursiveCharacterTextSplitter**: 여러 구분자를 순서대로 시도하는 재귀적 분할 (대부분의 경우 권장)
